# 08 — BiWave-UNet: Proposed Model Training

**Architecture:** UNet Encoder-Decoder + BiGRU + BiTCN + CBAM + DyT + Seq2Seg

**Innovation:** First multi-appliance NILM model combining UNet skip connections
with BiGRU+BiTCN temporal modeling, deployable on STM32MP2 NPU.

**Components combined from:**
- UNet-NILM (Faustine et al. 2020): encoder-decoder + skip connections
- BiGRU + BiTCN: bidirectional temporal modeling
- CBAM (Woo et al. 2018): channel + temporal attention
- DU-NILM (2024): seq2seg 96-point output
- NILMFormer (Petralia 2025): instance normalization
- Kelly & Knottenbelt (2015): state label filtering

**All preprocessing improvements:**
- DWT 4 sub-bands + instance norm
- Power scaling augmentation (0.7-1.3x)
- Gaussian noise (2%)
- Masked power loss
- Focal Loss alpha=0.75 gamma=2.0

**Parameters:** ~420K | INT8: 0.42 MB | Target: STM32MP2 NPU

**Author:** Chadha Jeddi — NILM Benchmarking Project

## 1. Setup

In [ ]:
import os, sys, glob
os.environ['PYTHONUNBUFFERED'] = '1'

REPO_DIR = '/kaggle/working/nilm-benchmarking'
SAVE_DIR = '/kaggle/working/nilm_results'
CKPT_DIR = f'{SAVE_DIR}/checkpoints'
RES_DIR  = f'{SAVE_DIR}/results'

for d in [SAVE_DIR, CKPT_DIR, RES_DIR]:
    os.makedirs(d, exist_ok=True)

if not os.path.exists(REPO_DIR):
    os.system(f'git clone https://github.com/chadhajeddi-ux/nilm-benchmarking {REPO_DIR}')
else:
    os.system(f'cd {REPO_DIR} && git pull')

os.chdir(REPO_DIR)
for p in ['src','models','models/baselines','models/proposed']:
    sys.path.insert(0, f'{REPO_DIR}/{p}')

for d in ['data/raw/UKDALE','data/raw/UK-DALE','data/processed']:
    os.makedirs(f'{REPO_DIR}/{d}', exist_ok=True)

ukdale_files = glob.glob('/kaggle/input/**/ukdale.h5', recursive=True)
if ukdale_files:
    for dst in ['data/raw/UKDALE/ukdale.h5','data/raw/UK-DALE/ukdale.h5']:
        if not os.path.exists(dst): os.symlink(ukdale_files[0], dst)
    print(f'OK UK-DALE: {ukdale_files[0]}')
else:
    print('ERROR: ukdale.h5 not found')

for f in glob.glob(f'{REPO_DIR}/data/processed/*.parquet'):
    os.remove(f)

os.system('pip install -q PyWavelets pyarrow h5py tqdm einops omegaconf torchinfo seaborn')

import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'PyTorch: {torch.__version__}')
print(f'Save dir: {SAVE_DIR}')


## 2. Imports & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import time, json, warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F

from config import (WINDOW_SIZE, INPUT_CHANNELS, N_APPLIANCES,
                    APPLIANCE_NAMES, APPLIANCES, SEED)
from preprocessing import load_ukdale_house, preprocess_house
from dataset import (load_clean_df, save_clean_df, split_train_val,
                     build_dataloaders)
from metrics import MetricsTracker, focal_loss
from train import EarlyStopping

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# BIWAVE-UNET CONFIGURATION
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
MODEL_NAME   = 'biwave'
LR           = 1e-4
WEIGHT_DECAY = 1e-2
DROPOUT      = 0.2
EPOCHS       = 100
PATIENCE     = 25
SCHED_PAT    = 15
SEG_SIZE     = 96

COLORS = {'kettle':'#D94040','fridge':'#2E9E5A','washing_machine':'#E8922A',
          'dishwasher':'#7B4FBF','microwave':'#CC3399'}
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print(f'Model: {MODEL_NAME} | LR={LR} | WD={WEIGHT_DECAY} | Seg={SEG_SIZE}')
print(f'Device: {DEVICE}')


## 3. Data Loading
Kelly state labels + instance normalization + seq2seg

In [ ]:
print('Loading UK-DALE House 1...')
cached = load_clean_df('UK-DALE', 1)
if cached is not None:
    clean_df = cached
else:
    raw_df = load_ukdale_house(house=1)
    clean_df = preprocess_house(raw_df)
    save_clean_df(clean_df, 'UK-DALE', 1)

print(f'Shape: {clean_df.shape}')
print(f'Duration: {(clean_df.index[-1]-clean_df.index[0]).days} days')

train_df, val_df = split_train_val(clean_df, val_fraction=0.15)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,}')

train_loader, val_loader, norm_stats = build_dataloaders(
    train_df, val_df,
    batch_size=256,
    train_stride=30,
    val_stride=480,
    num_workers=2,
    instance_norm=True,
    seg_size=SEG_SIZE,
)

x_s, yp_s, ys_s = next(iter(val_loader))
print(f'Batch: x={tuple(x_s.shape)} y_power={tuple(yp_s.shape)} y_state={tuple(ys_s.shape)}')
print(f'Train batches/epoch: {len(train_loader)}')

print('\nAppliance duty cycles:')
for a in APPLIANCE_NAMES:
    duty = train_df[f'{a}_state'].mean()*100
    print(f'  {a:<20} duty={duty:.2f}%')


## 4. Model Architecture
BiWave-UNet: Encoder(3L) → Bottleneck(BiGRU+BiTCN) → Decoder(3L) → Seq2Seg

In [ ]:
from torchinfo import summary
from proposed_model import BiWaveNILM

model = BiWaveNILM(
    in_channels=INPUT_CHANNELS,
    window_size=WINDOW_SIZE,
    n_appliances=N_APPLIANCES,
    seg_size=SEG_SIZE,
    dropout=DROPOUT,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,} | INT8: {n_params/1e6:.3f} MB')
print(f'Deployable on STM32MP2: {"YES" if n_params/1e6 < 16 else "NO"}')
print()
model.model_summary()
print()

# Verify forward pass
with torch.no_grad():
    test_x = torch.randn(2, INPUT_CHANNELS, WINDOW_SIZE).to(DEVICE)
    sp, cs, sg = model(test_x)
    print(f'Forward pass OK:')
    print(f'  seg_power: {tuple(sp.shape)}')
    print(f'  ctr_state: {tuple(cs.shape)}')
    print(f'  seg_gated: {tuple(sg.shape)}')


## 5. Training
All improvements: augmentation + masked power loss + focal loss + seq2seg

In [ ]:
CKPT_PATH = f'{CKPT_DIR}/{MODEL_NAME}_best.pth'
HIST_PATH = f'{RES_DIR}/{MODEL_NAME}_history.json'

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=SCHED_PAT)
early_stop = EarlyStopping(patience=PATIENCE)

app_max = {a: float(APPLIANCES[a]['max_power']) for a in APPLIANCE_NAMES}
tracker = MetricsTracker(APPLIANCE_NAMES, app_max)
history = {'epoch':[],'train_loss':[],'val_loss':[],
           'val_mr':[],'val_f1':[],'val_mae':[],'lr':[]}
best_mr, best_state, best_epoch = -float('inf'), None, 0
start_time = time.time()

print(f'Training {MODEL_NAME.upper()} | {n_params:,} params | seg2seg={SEG_SIZE}', flush=True)
print(f'Output: seg_power(B,5,{SEG_SIZE}) + center_state(B,5) + seg_gated(B,5,{SEG_SIZE})', flush=True)
print(f'LR={LR} | WD={WEIGHT_DECAY} | Dropout={DROPOUT}', flush=True)
print(f'Augmentation: power scaling (0.7-1.3) + noise (2%)', flush=True)
print(f'Loss: masked power + focal state (alpha=0.75) + gated', flush=True)
print(f'Saving to: {CKPT_PATH}', flush=True)
print('='*80, flush=True)

for epoch in range(1, EPOCHS+1):
    ep_start = time.time()

    # ---- TRAIN ----
    model.train()
    total_train_loss = 0.0
    n_batches = 0
    for x, y_power, y_state in train_loader:
        x       = x.to(DEVICE)         # (B, 6, 480)
        y_power = y_power.to(DEVICE)    # (B, 5, 96)
        y_state = y_state.to(DEVICE)    # (B, 5, 96)

        # ── Augmentation (training only) ──────────────────────
        scale = (0.7 + torch.rand(x.shape[0], 1, 1) * 0.6).to(DEVICE)
        x = x * scale
        y_power = (y_power * scale.squeeze(-1).unsqueeze(-1)).clamp(0, 1)
        x = x + torch.randn_like(x) * 0.02
        # ──────────────────────────────────────────────────────

        seg_p, ctr_s, seg_g = model(x)
        y_state_center = y_state[:, :, SEG_SIZE//2]  # (B, 5)

        # Masked power loss — full segment
        loss_power_all = F.smooth_l1_loss(seg_p, y_power)
        mask_seg = (y_state > 0.5)  # (B, 5, 96)
        if mask_seg.sum() > 0:
            loss_power_active = F.smooth_l1_loss(
                seg_p[mask_seg], y_power[mask_seg])
            loss_power = 0.5*loss_power_all + 0.5*loss_power_active
        else:
            loss_power = loss_power_all

        # State loss — center point
        loss_state = focal_loss(ctr_s, y_state_center, alpha=0.75, gamma=2.0)

        # Gated loss — full segment
        loss_gated = F.smooth_l1_loss(seg_g, y_power)

        loss = 1.0*loss_power + 2.0*loss_state + 0.5*loss_gated

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_train_loss += loss.item()
        n_batches += 1
    train_loss = total_train_loss / max(n_batches, 1)

    # ---- VALIDATE (no augmentation) ----
    model.eval()
    total_val_loss = 0.0
    v_batches = 0
    tracker.reset()
    with torch.no_grad():
        for x, yp, ys in val_loader:
            x  = x.to(DEVICE)
            yp = yp.to(DEVICE)   # (B, 5, 96)
            ys = ys.to(DEVICE)   # (B, 5, 96)

            seg_p, ctr_s, seg_g = model(x)
            ys_center = ys[:, :, SEG_SIZE//2]

            # Val loss
            loss_p_all = F.smooth_l1_loss(seg_p, yp)
            mask_v = (ys > 0.5)
            if mask_v.sum() > 0:
                loss_p_act = F.smooth_l1_loss(seg_p[mask_v], yp[mask_v])
                loss_vp = 0.5*loss_p_all + 0.5*loss_p_act
            else:
                loss_vp = loss_p_all
            loss = (loss_vp
                    + 2.0*focal_loss(ctr_s, ys_center, alpha=0.75, gamma=2.0)
                    + 0.5*F.smooth_l1_loss(seg_g, yp))
            total_val_loss += loss.item()
            v_batches += 1

            # Metrics: center point
            pp_c = seg_p[:, :, SEG_SIZE//2].cpu()
            yp_c = yp[:, :, SEG_SIZE//2].cpu()
            ys_c = ys_center.cpu()
            tracker.update(pp_c, yp_c, ctr_s.cpu(), ys_c)

    val_loss = total_val_loss / max(v_batches, 1)
    metrics  = tracker.compute()
    val_mr   = metrics['mean']['mr']
    val_f1   = metrics['mean']['f1']
    val_mae  = metrics['mean']['mae_w']
    cur_lr   = optimizer.param_groups[0]['lr']
    ep_time  = time.time() - ep_start

    for k, v in zip(
        ['epoch','train_loss','val_loss','val_mr','val_f1','val_mae','lr'],
        [epoch, train_loss, val_loss, val_mr, val_f1, val_mae, cur_lr]
    ):
        history[k].append(v)

    is_best = val_mr > best_mr
    if is_best:
        best_mr = val_mr; best_epoch = epoch
        best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
        torch.save({
            'model_name': MODEL_NAME,
            'model_state_dict': best_state,
            'best_epoch': best_epoch,
            'best_val_mr': best_mr,
            'n_params': n_params,
            'seg_size': SEG_SIZE,
            'norm_stats': {
                'agg_mean': norm_stats.agg_mean,
                'agg_std':  norm_stats.agg_std,
                'appliance_max': norm_stats.appliance_max,
            },
        }, CKPT_PATH)

    if epoch % 5 == 0 or is_best:
        with open(HIST_PATH, 'w') as f:
            json.dump({**history,
                'best_epoch': best_epoch, 'best_val_mr': best_mr,
                'model': MODEL_NAME,
                'training_time_seconds': time.time()-start_time}, f)

    star = ' *' if is_best else ''
    print(f'Ep {epoch:3d}/{EPOCHS} | L:{train_loss:.4f}/{val_loss:.4f} | '
          f'MR:{val_mr:.3f} F1:{val_f1:.3f} MAE:{val_mae:.1f}W | '
          f'LR:{cur_lr:.1e} | {ep_time:.0f}s{star}', flush=True)

    scheduler.step(val_mr)
    if early_stop.step(val_mr):
        print(f'Early stopping at epoch {epoch}', flush=True)
        break

total_time = time.time() - start_time
print(f'\nDone in {total_time/60:.1f} min | '
      f'Best epoch: {best_epoch} | Best MR: {best_mr:.4f}', flush=True)
print(f'Checkpoint: {CKPT_PATH}', flush=True)


## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
epochs_list = history['epoch']

axes[0,0].plot(epochs_list, history['train_loss'], color='#3366CC', label='Train')
axes[0,0].plot(epochs_list, history['val_loss'], color='#D94040', label='Val')
axes[0,0].axvline(best_epoch, color='gray', linestyle=':')
axes[0,0].set_title('Loss'); axes[0,0].legend(); axes[0,0].set_xlabel('Epoch')
axes[0,0].set_ylim(0.010, 0.055)

axes[0,1].plot(epochs_list, history['val_mr'], color='#2E9E5A', linewidth=2)
axes[0,1].axvline(best_epoch, color='gray', linestyle=':')
axes[0,1].axhline(best_mr, color='#2E9E5A', linestyle='--', alpha=0.4,
                  label=f'Best MR={best_mr:.3f}')
axes[0,1].set_title('Matching Ratio'); axes[0,1].legend(); axes[0,1].set_xlabel('Epoch')
axes[0,1].set_ylim(0, 1.0)

axes[1,0].plot(epochs_list, history['val_f1'], color='#E8922A', linewidth=2)
axes[1,0].axvline(best_epoch, color='gray', linestyle=':')
axes[1,0].set_title('F1 Score'); axes[1,0].set_xlabel('Epoch')
axes[1,0].set_ylim(0, 1.0)

axes[1,1].plot(epochs_list, history['val_mae'], color='#7B4FBF', linewidth=2)
axes[1,1].axvline(best_epoch, color='gray', linestyle=':')
axes[1,1].set_title('MAE (Watts)'); axes[1,1].set_xlabel('Epoch')
axes[1,1].set_ylim(0, 25)

plt.suptitle(f'BiWave-UNet Training Curves\n'
             f'Best: ep{best_epoch} | MR={best_mr:.3f} | '
             f'F1={max(history["val_f1"]):.3f} | '
             f'MAE={min(history["val_mae"]):.1f}W', fontsize=14)
plt.tight_layout()
plt.savefig(f'{RES_DIR}/{MODEL_NAME}_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Final Evaluation (House 1 Validation)

In [ ]:
model.load_state_dict(best_state)
model.to(DEVICE); model.eval()

tracker2 = MetricsTracker(APPLIANCE_NAMES, app_max)
tracker2.reset()
all_pp, all_tp, all_ps, all_ts = [], [], [], []

with torch.no_grad():
    for x, yp, ys in val_loader:
        seg_p, ctr_s, seg_g = model(x.to(DEVICE))
        pp_c = seg_p[:, :, SEG_SIZE//2].cpu()
        yp_c = yp[:, :, SEG_SIZE//2]
        ys_c = ys[:, :, SEG_SIZE//2]
        tracker2.update(pp_c, yp_c, ctr_s.cpu(), ys_c)
        all_pp.append(pp_c.numpy())
        all_tp.append(yp_c.numpy())
        all_ps.append((torch.sigmoid(ctr_s.cpu())>=0.5).float().numpy())
        all_ts.append(ys_c.numpy())

final_metrics = tracker2.compute()
pred_power = np.concatenate(all_pp)
true_power = np.concatenate(all_tp)
pred_state = np.concatenate(all_ps)
true_state = np.concatenate(all_ts)

print(f'\n{"="*80}')
print(f'FINAL RESULTS — BiWave-UNet (epoch {best_epoch})')
print(f'{"="*80}')
tracker2.print_table(final_metrics)

tracker2.save_json(f'{RES_DIR}/{MODEL_NAME}_metrics.json',
                   final_metrics, model_name=MODEL_NAME)
tracker2.to_dataframe(final_metrics).to_csv(
    f'{RES_DIR}/{MODEL_NAME}_metrics.csv', index=False)


## 8. Disaggregation Visualization

In [ ]:
N_SHOW = 1500
best_start, best_active = 0, 0
for s in range(0, min(len(true_power)-N_SHOW, 50000), N_SHOW):
    active = sum(true_state[s:s+N_SHOW, i].max() > 0 for i in range(N_APPLIANCES))
    if active > best_active:
        best_active = active; best_start = s
        if active == N_APPLIANCES: break

fig, axes = plt.subplots(N_APPLIANCES, 1, figsize=(16, 3.5*N_APPLIANCES), sharex=True)
t = range(best_start, best_start+N_SHOW)
for i, a in enumerate(APPLIANCE_NAMES):
    ax = axes[i]
    max_w = APPLIANCES[a]['max_power']
    tw = true_power[best_start:best_start+N_SHOW, i] * max_w
    pw = pred_power[best_start:best_start+N_SHOW, i] * max_w
    ax.plot(t, tw, color='#D94040', linewidth=1.2, label='Ground truth')
    ax.plot(t, pw, color='#2196F3', linewidth=1.2, label='BiWave-UNet')
    ax.set_ylabel(a.replace('_',' ').title() + '\n(W)', fontsize=10)
    mr_v = final_metrics[a]['mr']; f1_v = final_metrics[a]['f1']
    mae_v = final_metrics[a]['mae_w']
    ax.text(0.01, 0.92, f'MR={mr_v:.3f}  F1={f1_v:.3f}  MAE={mae_v:.1f}W',
            transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.85))
    if i==0: ax.legend(loc='upper right', fontsize=8, ncol=2)
axes[-1].set_xlabel('Timestep (x6 seconds)')
plt.suptitle('BiWave-UNet — Disaggregation Results', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{RES_DIR}/{MODEL_NAME}_disaggregation.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, N_APPLIANCES, figsize=(4*N_APPLIANCES, 4))
for i, a in enumerate(APPLIANCE_NAMES):
    cm = confusion_matrix(true_state[:,i], pred_state[:,i], labels=[0,1])
    cm_norm = cm.astype('float')/(cm.sum(axis=1,keepdims=True)+1e-8)
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=['OFF','ON'], yticklabels=['OFF','ON'],
                ax=axes[i], cbar=i==N_APPLIANCES-1, vmin=0, vmax=1)
    f1_v = final_metrics[a]['f1']
    prec = final_metrics[a]['precision']; rec = final_metrics[a]['recall']
    axes[i].set_title(f'{a.replace("_"," ").title()}\nF1={f1_v:.3f} P={prec:.3f} R={rec:.3f}', fontsize=9)
    axes[i].set_xlabel('Predicted'); axes[i].set_ylabel('Actual' if i==0 else '')
plt.suptitle('BiWave-UNet — Confusion Matrices', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RES_DIR}/{MODEL_NAME}_confusion.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. Per-Appliance Bar Charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = [COLORS[a] for a in APPLIANCE_NAMES]
x_pos = np.arange(N_APPLIANCES)
for ax, metric, label, fmt in zip(axes,
    ['f1','mae_w','mr'], ['F1 Score','MAE (W)','Matching Ratio'],
    ['{:.3f}','{:.1f}','{:.3f}']):
    vals = [final_metrics[a][metric] for a in APPLIANCE_NAMES]
    bars = ax.bar(x_pos, vals, color=colors, alpha=0.85)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(APPLIANCE_NAMES, rotation=25, ha='right')
    ax.set_title(label)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, val*1.02, fmt.format(val),
                ha='center', fontsize=9)
plt.suptitle('BiWave-UNet — Per-Appliance Results', fontsize=13)
plt.tight_layout()
plt.savefig(f'{RES_DIR}/{MODEL_NAME}_bar_charts.png', dpi=150, bbox_inches='tight')
plt.show()


## 11. Cross-House Evaluation (H2 + H5)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from dwt import dwt_transform

class CrossHouseDataset(Dataset):
    """Seq2seg dataset for cross-house evaluation."""
    def __init__(self, df, stride=480, seg_size=96):
        self.df = df
        self.seg_size = seg_size
        self.starts = list(range(0, len(df)-WINDOW_SIZE+1, stride))
    def __len__(self): return len(self.starts)
    def __getitem__(self, i):
        s = self.starts[i]; c = s + WINDOW_SIZE//2
        seg_s = c - self.seg_size//2; seg_e = c + self.seg_size//2
        window = self.df['aggregate'].values[s:s+WINDOW_SIZE].astype(np.float32)
        w_mean = window.mean(); w_std = window.std() + 1e-8
        x = dwt_transform((window - w_mean) / w_std)
        hour = self.df.index[c].hour
        sin_h = np.full(WINDOW_SIZE, np.sin(2*np.pi*hour/24), dtype=np.float32)
        cos_h = np.full(WINDOW_SIZE, np.cos(2*np.pi*hour/24), dtype=np.float32)
        x = np.concatenate([x, sin_h[None], cos_h[None]], axis=0)
        y_power = np.array([np.nan_to_num(
            self.df[a].values[seg_s:seg_e]/APPLIANCES[a]['max_power'], nan=0.0)
            for a in APPLIANCE_NAMES], dtype=np.float32).clip(0,1)
        y_state = np.array([np.nan_to_num(
            self.df[f'{a}_state'].values[seg_s:seg_e], nan=0.0)
            for a in APPLIANCE_NAMES], dtype=np.float32)
        return torch.tensor(x), torch.tensor(y_power), torch.tensor(y_state)

model.load_state_dict(best_state); model.to(DEVICE); model.eval()

for house_num in [2, 5]:
    print(f'\nLoading UK-DALE House {house_num}...', flush=True)
    cached_h = load_clean_df('UK-DALE', house_num)
    if cached_h is not None:
        h_df = cached_h
    else:
        raw_h = load_ukdale_house(house=house_num)
        h_df = preprocess_house(raw_h)
        save_clean_df(h_df, 'UK-DALE', house_num)
    print(f'House {house_num}: {len(h_df):,} rows')

    h_loader = DataLoader(
        CrossHouseDataset(h_df, stride=480, seg_size=SEG_SIZE),
        batch_size=256, shuffle=False, num_workers=2)

    h_tracker = MetricsTracker(APPLIANCE_NAMES, app_max)
    h_tracker.reset()
    with torch.no_grad():
        for x, yp, ys in h_loader:
            seg_p, ctr_s, seg_g = model(x.to(DEVICE))
            pp_c = seg_p[:, :, SEG_SIZE//2].cpu()
            yp_c = yp[:, :, SEG_SIZE//2]
            ys_c = ys[:, :, SEG_SIZE//2]
            h_tracker.update(pp_c, yp_c, ctr_s.cpu(), ys_c)

    h_metrics = h_tracker.compute()
    print(f'\n{"="*70}')
    print(f'HOUSE {house_num} RESULTS (unseen) — BiWave-UNet')
    print(f'{"="*70}')
    h_tracker.print_table(h_metrics)

    with open(f'{RES_DIR}/{MODEL_NAME}_h{house_num}_metrics.json', 'w') as f:
        json.dump(h_metrics, f, indent=2)
    print(f'Saved: {MODEL_NAME}_h{house_num}_metrics.json')


## 12. Save All Results + Download

In [ ]:
# Save norm stats and final history
norm_stats.save(f'{RES_DIR}/{MODEL_NAME}_norm_stats.json')
with open(HIST_PATH, 'w') as f:
    json.dump({**history,
        'best_epoch': best_epoch, 'best_val_mr': best_mr,
        'model': MODEL_NAME,
        'training_time_seconds': total_time}, f, indent=2)

# Verify all files
import os, zipfile
expected = [
    f'{CKPT_DIR}/{MODEL_NAME}_best.pth',
    f'{RES_DIR}/{MODEL_NAME}_history.json',
    f'{RES_DIR}/{MODEL_NAME}_metrics.json',
    f'{RES_DIR}/{MODEL_NAME}_metrics.csv',
    f'{RES_DIR}/{MODEL_NAME}_h2_metrics.json',
    f'{RES_DIR}/{MODEL_NAME}_h5_metrics.json',
    f'{RES_DIR}/{MODEL_NAME}_norm_stats.json',
    f'{RES_DIR}/{MODEL_NAME}_training_curves.png',
    f'{RES_DIR}/{MODEL_NAME}_disaggregation.png',
    f'{RES_DIR}/{MODEL_NAME}_confusion.png',
    f'{RES_DIR}/{MODEL_NAME}_bar_charts.png',
]

print('Files verification:')
missing = []
for fp in expected:
    if os.path.exists(fp):
        size = os.path.getsize(fp)/1e6
        print(f'  OK   {os.path.basename(fp)}: {size:.2f} MB')
    else:
        print(f'  MISS {os.path.basename(fp)}')
        missing.append(fp)

if not missing:
    zip_path = '/kaggle/working/biwave_unet_complete.zip'
    with zipfile.ZipFile(zip_path, 'w') as zf:
        for root, dirs, files in os.walk('/kaggle/working/nilm_results'):
            for file in files:
                if MODEL_NAME in file:
                    full = os.path.join(root, file)
                    zf.write(full, file)
    size = os.path.getsize(zip_path)/1e6
    print(f'\nZIP: biwave_unet_complete.zip ({size:.1f} MB)')
    print('Download from Output tab NOW')
else:
    print(f'\nMISSING {len(missing)} files!')

print(f'\n{"="*70}')
print(f'BiWave-UNet | Best MR: {best_mr:.4f} | Epoch: {best_epoch}')
print(f'Parameters: {n_params:,} | INT8: {n_params/1e6:.3f} MB')
print(f'Training time: {total_time/60:.1f} min')
print(f'{"="*70}')
